# S2 — Faciès pétroliers et validation par puits

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic1/notebooks/IC1_S2_facies_par_puits.ipynb)
> Données : dossier `IC1_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['facies_diagraphies.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic1/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict, learning_curve, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
fa = pd.read_csv('facies_diagraphies.csv')
X = fa[['GR_api', 'RHOB_gcc', 'NPHI_vv', 'logRT_ohmm', 'PE_be']]; y, puits = fa['facies'], fa['puits']

## 1. k-fold aléatoire vs GroupKFold par puits

In [ ]:
modeles = {'knn': make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
 'logit': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
 'rf': make_pipeline(StandardScaler(), RandomForestClassifier(300, min_samples_leaf=3, random_state=0))}
for nom, m in modeles.items():
    a = cross_val_score(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
    b = cross_val_score(m, X, y, cv=GroupKFold(6), groups=puits).mean()
    print(nom, round(a, 3), round(b, 3))

## 2. Matrice de confusion (puits jamais vus)

In [ ]:
pred = cross_val_predict(modeles['rf'], X, y, cv=GroupKFold(6), groups=puits)
ConfusionMatrixDisplay.from_predictions(y, pred, normalize='true', values_format='.2f'); plt.show()

## Bonus
Normaliser GR par puits (`fa.groupby('puits')['GR_api'].transform(lambda s: (s - s.mean()) / s.std())`) et refaire la validation par puits.